In [53]:
# LIBRARIES

import pandas as pd
from pathlib import Path

In [54]:
# PATHS

INPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_nlp.csv"
)

OUTPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry.csv"
)

df = pd.read_csv(INPUT_FILE)

print("Total jobs:", len(df))
print("Columns:", df.columns.tolist())

Total jobs: 19207
Columns: ['job_id', 'source_id', 'title', 'company', 'location', 'description', 'salary_min', 'salary_max', 'salary_avg', 'created', 'contract_type', 'contract_time', 'url', 'source', 'title_clean', 'description_clean']


In [55]:
def classify_industry(title, description):

    title = str(title).lower().strip()
    description = str(description).lower().strip()

# STRONG TITLE-BASED RULES

    TITLE_RULES = {

        "IT and Technology": [
            "software developer",
            "software engineer",
            "web developer",
            "data scientist",
            "data analyst",
            "machine learning",
            "cybersecurity",
            "information security",
            "it support",
            "it manager",
            "cloud engineer",
            "devops",
            "programmer",
            "c# developer",
            "python developer",
            "java developer",
            "sql developer",
            "network engineer",
            "systems administrator",
            "service integration manager",
            "technical delivery manager"
        ],

        "Healthcare": [
            "pharmacist",
            "pharmacy assistant",
            "pharmacy colleague",
            "dispenser",
            "nurse",
            "nursing",
            "doctor",
            "medical",
            "healthcare",
            "clinical",
            "dentist",
            "dental",
            "physiotherapist"
        ],

        "Finance and Banking": [
            "investment banker",
            "investment banking analyst",
            "financial analyst",
            "finance analyst",
            "financial controller",
            "accountant",
            "accounting",
            "private equity",
            "asset management",
            "wealth management",
            "portfolio manager",
            "treasury",
            "investment analyst",
            "fund manager",
            "payroll administrator"
        ],

        "Insurance": [
            "insurance analyst",
            "insurance broker",
            "insurance adviser",
            "underwriter",
            "underwriting",
            "actuary",
            "actuarial",
            "claims handler"
        ],

        "Retail": [
            "store manager",
            "store assistant",
            "store colleague",
            "service colleague",
            "sales assistant",
            "retail assistant",
            "retail manager",
            "shop assistant",
            "supermarket",
            "cashier",
            "merchandiser",
            "kitchen sales designer"
        ],

        "Hospitality and Food Services": [
            "restaurant manager",
            "restaurant general manager",
            "restaurant supervisor",
            "chef",
            "sous chef",
            "waiter",
            "waitress",
            "barista",
            "hotel manager",
            "housekeeping",
            "catering manager",
            "food services"
        ],

        "Education": [
            "teacher",
            "teaching assistant",
            "lecturer",
            "professor",
            "tutor",
            "education officer",
            "school teacher",
            "learning support"
        ],

        "Construction": [
            "electrician",
            "plumber",
            "bricklayer",
            "carpenter",
            "builder",
            "construction manager",
            "site manager",
            "site supervisor",
            "quantity surveyor"
        ],

        "Engineering": [
            "mechanical engineer",
            "electrical engineer",
            "civil engineer",
            "chemical engineer",
            "design engineer",
            "maintenance engineer",
            "engineering manager"
        ],

        "Transport and Logistics": [
            "delivery driver",
            "customer delivery driver",
            "hgv driver",
            "lgv driver",
            "warehouse operative",
            "warehouse manager",
            "logistics manager",
            "supply chain manager",
            "forklift operator",
            "reach truck driver",
            "courier",
            "fleet manager"
        ],

        "Legal": [
            "solicitor",
            "barrister",
            "paralegal",
            "legal assistant",
            "legal counsel",
            "lawyer"
        ],

        "Real Estate": [
            "estate agent",
            "property manager",
            "property consultant",
            "letting agent",
            "real estate"
        ],

        "Automotive": [
            "automotive technician",
            "vehicle technician",
            "vehicle mechanic",
            "car mechanic",
            "motor vehicle technician",
            "automotive engineer"
        ],

        "Manufacturing": [
            "production operative",
            "production manager",
            "manufacturing operative",
            "machine operator",
            "assembly operative",
            "factory worker"
        ],

        "Marketing and Advertising": [
            "marketing manager",
            "marketing executive",
            "digital marketing",
            "seo specialist",
            "content marketer",
            "brand manager"
        ],

        "Human Resources": [
            "hr manager",
            "hr advisor",
            "hr assistant",
            "human resources",
            "recruiter",
            "recruitment consultant",
            "talent acquisition"
        ]
    }


# CHECK JOB TITLE FIRST
    
    title_matches = {}

    for industry, keywords in TITLE_RULES.items():

        matches = []

        for keyword in keywords:

            if keyword in title:
                matches.append(keyword)

        if matches:
            title_matches[industry] = matches


# STRONG TITLE MATCH WINS

    if title_matches:

        best_industry = max(
            title_matches,
            key=lambda industry: max(
                len(keyword)
                for keyword in title_matches[industry]
            )
        )

        return best_industry


# DESCRIPTION FALLBACK
# Conservative fallback.
# If the job title does not provide a strong
# classification signal, return Other.

    return "Other"

In [57]:
# FULL DATASET INDUSTRY CLASSIFICATION

print("Starting industry classification...")
print("Total jobs:", len(df))

df["industry"] = df.apply(
    lambda row: classify_industry(
        row["title"],
        row["description_clean"]
    ),
    axis=1
)

print("\nIndustry classification completed.")

print("\nIndustry distribution:")
print(df["industry"].value_counts())

Starting industry classification...
Total jobs: 19207

Industry classification completed.

Industry distribution:
industry
Other                            13651
Engineering                       1043
Education                          912
Finance and Banking                664
Manufacturing                      596
Construction                       564
Transport and Logistics            381
IT and Technology                  370
Hospitality and Food Services      249
Human Resources                    155
Healthcare                         142
Legal                              136
Automotive                         124
Retail                             113
Marketing and Advertising           63
Real Estate                         22
Insurance                           22
Name: count, dtype: int64


In [58]:
# SAVE INDUSTRY CLASSIFIED DATASET

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved successfully:")
print(OUTPUT_FILE)

Saved successfully:
C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry.csv


In [59]:
# VERIFY

check_df = pd.read_csv(OUTPUT_FILE)

print("Rows:", len(check_df))
print("Columns:", len(check_df.columns))
print("Missing industries:", check_df["industry"].isna().sum())

Rows: 19207
Columns: 17
Missing industries: 0
